In [ ]:
#Old Notebook used before QAOA class was developed.

In [ ]:
import rustworkx as rx
from rustworkx.visualization import mpl_draw as draw_graph
import numpy as np
import random
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.simplefilter("ignore", UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

from qiskit import transpile
from qiskit.circuit import Parameter,ParameterExpression
from qiskit_algorithms import NumPyMinimumEigensolver
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import QAOAAnsatz
from qiskit_ibm_runtime import Session, EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime.fake_provider import FakeMumbaiV2
from qiskit.converters import circuit_to_dag, dag_to_circuit

import sys
sys.path.append("../")
from clapton.clapton import claptonize
from clapton.circuit_manipulation import transform_to_allowed_gates,qiskit_to_stim, modify_circuit, multi_angle_qaoa_circuit, transform_qiskit_to_stim,generate_qiskit_param_map
from testing_scripts.graphs_utils import generate_random_complete_graph,generate_k_regular_graph,compute_optimal_max_cut,build_max_cut_paulis
from testing_scripts.qaoa_utils import QAOASolver,evaluate_energy

In [ ]:
n = 10

k=3
# G = generate_k_regular_graph(num_vertices=n, k=k, weighted=True,seed=True)

G=generate_random_complete_graph(num_vertices=n, weighted=True, seed=True)
draw_graph(G, node_size=600, with_labels=True)

In [ ]:
# Optimal Maxcut Calculation 
optimal_max_cut_val = compute_optimal_max_cut(G)
print(f"Optimal Max Cut Value : {optimal_max_cut_val}")

In [ ]:
max_cut_paulis = build_max_cut_paulis(G)

cost_hamiltonian = SparsePauliOp.from_list(max_cut_paulis)
print("Cost Function Hamiltonian:", cost_hamiltonian)
paulis,coeffs = cost_hamiltonian.paulis.to_labels(),cost_hamiltonian.coeffs.real

In [ ]:
# Need to reverse the Paulis for stim adapation 
reversed_paulis = [p[::-1] for p in paulis]
print(reversed_paulis)

In [ ]:
reps = 2
circuit = multi_angle_qaoa_circuit(n,G ,reps)

In [ ]:
stim_circ,pcirc = transform_qiskit_to_stim(circuit)
stim_circ.stim_circuit().diagram()

In [ ]:
param_map = generate_qiskit_param_map(pcirc)
param_map

In [ ]:
stim_circ.define_parameter_map(param_map)

## CAFQA to find best initial params

In [ ]:
# we can perform CAFQA by using the main optimization function "claptonize"

ks_best, _, energy_best = claptonize(
    reversed_paulis,    #to respect stim ordering for hamiltonian
    coeffs,
    stim_circ,
    n_proc=4,           # total number of processes in parallel
    n_starts=4,         # number of random genetic algorithm starts in parallel
    n_rounds=1,         # number of budget rounds, if None it will terminate itself
    callback=print,     # callback for internal parameter (#iteration, energies, ks) processing
    budget=10          # budget per genetic algorithm instance
)

In [ ]:
energy_best

In [ ]:
stim_circ.assign(ks_best)
stim_circ.stim_circuit().diagram()

In [ ]:
ordered_params = [param.name for param in pcirc.parameters]
angle_multipliers = [-np.pi/4 if 'gamma' in param else np.pi/4 for param in ordered_params]

init_params = [param * (np.pi/2) for param,multiplier in zip(ks_best,angle_multipliers)] #This has to be in the order we come across the gates.

In [ ]:
energy = evaluate_energy(pcirc, cost_hamiltonian, init_params)
print(energy)

In [ ]:
objective_func_vals = []

def cost_func_estimator(params, ansatz, hamiltonian, estimator):
    global objective_func_vals

    # transform the observable defined on virtual qubits to
    # an observable defined on all physical qubits
    isa_hamiltonian = hamiltonian.apply_layout(ansatz.layout)

    pub = (ansatz, isa_hamiltonian, params)
    job = estimator.run([pub])

    results = job.result()[0]
    cost = results.data.evs
    
    objective_func_vals.append(cost)

    return cost

In [ ]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from scipy.optimize import minimize

noise_model = NoiseModel()
noisy_backend = FakeMumbaiV2() # The fake backend
noise_model = NoiseModel.from_backend(noisy_backend) 


backend = AerSimulator(method='statevector',noise_model=noise_model)
estimator = Estimator(mode=backend)
random_params = np.random.random(len(init_params))
result = minimize(
    cost_func_estimator,
    init_params,
    args=(pcirc, cost_hamiltonian, estimator),
    method="COBYLA",
    tol=1e-4,
    options = {'maxiter':10}
)
print(result)

In [ ]:
result.fun

In [ ]:
# Solve with classical Eigensolver for comparison
eigensolver = NumPyMinimumEigensolver()
exact_solution = eigensolver.compute_minimum_eigenvalue(cost_hamiltonian).eigenvalue.real
print("Exact Energy from Eigensolver:", exact_solution)

In [ ]:
random_energies = [evaluate_energy(pcirc, cost_hamiltonian, np.random.random(len(ks_best))) for _ in range(1)]
min_energy = min(random_energies)
print(f"Minimum Energy found with Random initialization over 100 runs: {min_energy}")

In [ ]:
rounded_angles = np.random.choice(np.arange(-np.pi, np.pi + np.pi/8, np.pi/8), len(ks_best))
rounded_energies = [evaluate_energy(pcirc, cost_hamiltonian, rounded_angles) for _ in range(1)]
print(f"Minimum Energy found with Angle Rounding over 100 runs: {min(rounded_energies)}")

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(objective_func_vals)
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.show()

In [ ]:
optimized_circuit = pcirc.assign_parameters(result.x)
optimized_circuit.measure_all()
optimized_circuit.draw('mpl', fold=False, idle_wires=False)

In [ ]:
sampler = Sampler(mode=backend)
sampler.options.default_shots = 10000

pub= (optimized_circuit, )
job = sampler.run([pub], shots=int(1e4))
counts_int = job.result()[0].data.meas.get_int_counts()
counts_bin = job.result()[0].data.meas.get_counts()
shots = sum(counts_int.values())
final_distribution_int = {key: val/shots for key, val in counts_int.items()}
final_distribution_bin = {key: val/shots for key, val in counts_bin.items()}
print(final_distribution_int)

In [ ]:
# auxiliary functions to sample most likely bitstring
def to_bitstring(integer, num_bits):
    result = np.binary_repr(integer, width=num_bits)
    return [int(digit) for digit in result]

keys = list(final_distribution_int.keys())
values = list(final_distribution_int.values())
most_likely = keys[np.argmax(np.abs(values))]
most_likely_bitstring = to_bitstring(most_likely, len(G))
most_likely_bitstring.reverse()

print("Result bitstring:", most_likely_bitstring)

In [ ]:
matplotlib.rcParams.update({"font.size": 10})
final_bits = final_distribution_bin
values = np.abs(list(final_bits.values()))
top_4_values = sorted(values, reverse=True)[:4]
positions = []
for value in top_4_values:
    positions.append(np.where(values == value)[0][0])
fig = plt.figure(figsize=(11, 6))
ax = fig.add_subplot(1, 1, 1)
plt.xticks(rotation=45)
plt.title("Result Distribution")
plt.xlabel("Bitstrings (reversed)")
plt.ylabel("Probability")
ax.bar(list(final_bits.keys()), list(final_bits.values()), color="tab:grey")
for p in positions:
    ax.get_children()[int(p)].set_color("tab:purple")
plt.show()

In [ ]:
# auxiliary function to plot graphs
def plot_result(G, x):
    colors = ["tab:grey" if i == 0 else "tab:purple" for i in x]
    pos, default_axes = rx.spring_layout(G), plt.axes(frameon=True)
    rx.visualization.mpl_draw(G, node_color=colors, node_size=100, alpha=0.8, pos=pos)

plot_result(G, most_likely_bitstring)

In [ ]:
from typing import Sequence
def evaluate_sample(x: Sequence[int], graph: rx.PyGraph) -> float:
    assert len(x) == len(list(graph.nodes())), "The length of x must coincide with the number of nodes in the graph."
    return sum(w*(x[u] * (1 - x[v]) + x[v] * (1 - x[u])) for u, v,w in list(graph.weighted_edge_list()))


cut_value= evaluate_sample(most_likely_bitstring, G)
print('The value of the cut is:', cut_value)

In [ ]:
top_4_values